# 01 — Quick Start: Predict with a Published Model

This notebook shows the fastest path to predictions: load a published
checkpoint from the HuggingFace Hub and run inference on SMILES strings.
No training, no custom data — just install, load, and predict.

**What you'll learn**
- Load a `CageFusionPipeline` from the Hub
- Run single-string, list, and DataFrame inference
- Inspect prediction columns and confidence thresholds

In [ ]:
# Install (skip if already installed)
# !pip install cage_fusion

In [1]:
from cage_fusion import CageFusionPipeline

# Downloads on first call, cached in ~/.cache/huggingface/hub afterwards.
# Swap the repo ID for any other published cage-fusion checkpoint.
pipe = CageFusionPipeline.from_pretrained("sidxz/cage-fusion-nuisance")
print("Pipeline ready.  Tasks:", pipe.tasks)

[02/28/26 15:24:58] INFO     HTTP Request: GET                                                                     
                             https://huggingface.co/api/models/sidxz/cage-fusion-nuisance/revision/main "HTTP/1.1  
                             404 Not Found"

FileNotFoundError: 'sidxz/cage-fusion-nuisance' is not a local directory and could not be downloaded from the Hub: 404 Client Error. (Request ID: Root=1-69a35d2a-489dfa37625eda4e64c8f5c9;92febc71-ec73-41d8-9bb7-461cf53046e7)

Repository Not Found for url: https://huggingface.co/api/models/sidxz/cage-fusion-nuisance/revision/main.
Please make sure you specified the correct `repo_id` and `repo_type`.
If you are trying to access a private or gated repo, make sure you are authenticated. For more details, see https://huggingface.co/docs/huggingface_hub/authentication

## Single SMILES → dict

In [ ]:
aspirin = "CC(=O)Oc1ccccc1C(=O)O"

result = pipe(aspirin)
print(result)

The output dict contains:
- `SMILES` — input SMILES string
- `<task>` — probability score for each task
- `pred_class_<task>` — 0/1 prediction using the best threshold

## List of SMILES → list of dicts

In [ ]:
smiles_list = [
    "CC(=O)Oc1ccccc1C(=O)O",   # aspirin
    "c1ccc2ccccc2c1",           # naphthalene
    "CN1C=NC2=C1C(=O)N(C(=O)N2C)C",  # caffeine
    "CC12CCC3C(C1CCC2O)CCC4=CC(=O)CCC34C",  # testosterone
]

results = pipe(smiles_list)
for r in results:
    print(r["SMILES"][:30].ljust(32), {k: round(v, 3) for k, v in r.items() if k in pipe.tasks})

## DataFrame → DataFrame

In [ ]:
import pandas as pd

df = pd.DataFrame({"SMILES": smiles_list, "name": ["aspirin", "naphthalene", "caffeine", "testosterone"]})

out = pipe(df)
out[["SMILES"] + pipe.tasks + [f"pred_class_{t}" for t in pipe.tasks]]

## Inspect pipeline metadata

In [ ]:
print("Tasks           :", pipe.tasks)
print("Best thresholds :", pipe.best_thresholds.round(3))
print("Device          :", pipe.device)
print()
print("Config:")
print(f"  attn_mode     = {pipe.config.attn_mode}")
print(f"  num_labels    = {pipe.config.num_labels}")
print(f"  embedding_dim = {pipe.config.embedding_dim}")
print(f"  graph_dim     = {pipe.config.graph_dim}")
print(f"  model_task    = {pipe.config.model_task}")

## Next steps

| Notebook | Topic |
|---|---|
| `02_moleculenet_benchmarks.ipynb` | Evaluate on public datasets |
| `03_train_custom_data.ipynb` | Train from your own CSV |
| `04_finetune_from_pretrained.ipynb` | Adapt published weights to a new task |
| `05_inference_and_explainability.ipynb` | Saliency and attention visualisation |